# 06C — Frozen LightGBM temporal decision-policy audit

This notebook does **not** retrain or replace the frozen 05C LightGBM model. It uses the completed 05H expanding-window predictions to test whether classification thresholds learned from prior forward-prediction years transfer to a later year.

The analysis separates probability-model performance from operational decisions. Thresholds are reported as scenario-specific policies, never as a universal optimum.

## Leakage-safe temporal design

- The score is `pred_forward_fixed_05c` from 05H. For each year, that score came from the fixed 05C LightGBM trained only on earlier years.
- 2021 is a warm-up year: it supplies the first leakage-safe historical predictions but receives no data-selected threshold.
- For test year 2022, thresholds use only 2021 forward predictions; for 2023 they use 2021–2022; for 2024 they use 2021–2023.
- A fixed 0.5 policy is included without selection.
- Within-year oracle thresholds are diagnostic only and are never presented as deployable results.
- Explicit false-positive/false-negative cost ratios are sensitivity scenarios, not asserted operational costs.

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "06C_V1_LIGHTGBM_TEMPORAL_DECISION_POLICY"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))
FORWARD_PREDICTIONS_PATH = PROJECT_ROOT / "data" / "processed" / "forward_temporal_generalization_v1" / "FORWARD_TEMPORAL_PREDICTIONS_V1.parquet"
UPSTREAM_CHECKS_PATH = PROJECT_ROOT / "outputs" / "forward_temporal_generalization_v1" / "FINAL_FORWARD_TEMPORAL_CHECKS_V1.csv"
UPSTREAM_CONTRACT_PATH = PROJECT_ROOT / "outputs" / "forward_temporal_generalization_v1" / "FORWARD_TEMPORAL_MODEL_CONTRACT_V1.json"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "lightgbm_temporal_decision_policy_v1"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "lightgbm_temporal_decision_policy_v1"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
for path in [OUTPUT_ROOT, PROCESSED_ROOT, FIGURE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

for path in [FORWARD_PREDICTIONS_PATH, UPSTREAM_CHECKS_PATH, UPSTREAM_CONTRACT_PATH]:
    assert path.exists(), path

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
}
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 360)

FORWARD_YEARS = [2021, 2022, 2023, 2024]
EVALUATION_YEARS = [2022, 2023, 2024]
EXPECTED_YEAR_UNITS = {2021: 101_517, 2022: 31_232, 2023: 10_467, 2024: 112_331}
EXPECTED_YEAR_INCIDENTS = {2021: 87, 2022: 54, 2023: 47, 2024: 104}
SCORE_COLUMN = "pred_forward_fixed_05c"
THRESHOLDS = np.round(np.arange(0.01, 1.00, 0.01), 2)
DECISION_THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.01), 2)
PPV_TARGET = 0.85
COST_SCENARIOS = {
    "COST_FP1_FN4": (1.0, 4.0),
    "COST_FP1_FN1": (1.0, 1.0),
    "COST_FP4_FN1": (4.0, 1.0),
    "COST_FP9_FN1": (9.0, 1.0),
}
POLICIES = [
    "FIXED_0_5",
    "MAX_PAST_MIXED_INCIDENT_BA",
    "PAST_PPV_AT_LEAST_0_85",
    *COST_SCENARIOS.keys(),
]
POLICY_LABELS = {
    "FIXED_0_5": "Fixed 0.5",
    "MAX_PAST_MIXED_INCIDENT_BA": "Max past mixed-incident BA",
    "PAST_PPV_AT_LEAST_0_85": "Past PPV ≥ 0.85",
    "COST_FP1_FN4": "Cost FP:FN = 1:4",
    "COST_FP1_FN1": "Cost FP:FN = 1:1",
    "COST_FP4_FN1": "Cost FP:FN = 4:1",
    "COST_FP9_FN1": "Cost FP:FN = 9:1",
}
COLORS = {policy: color for policy, color in zip(POLICIES, ["#4c72b0", "#55a868", "#c44e52", "#8172b2", "#937860", "#da8bc3", "#8c8c8c"])}

print("=" * 108)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 108)
print("No model fitting: using saved 05H forward predictions.")

In [ ]:
with UPSTREAM_CONTRACT_PATH.open("r", encoding="utf-8") as handle:
    upstream_contract = json.load(handle)
upstream_checks = pd.read_csv(UPSTREAM_CHECKS_PATH)
upstream_pass = upstream_checks["pass"].astype(str).str.lower().isin(["true", "1"]).all()
assert upstream_pass, "05H upstream checks did not all pass"
assert upstream_contract["feature_set"] == "E_FULL_EXTENDED"
assert upstream_contract["feature_count"] == 803
assert upstream_contract["classification_threshold"] is None
assert upstream_contract["posthoc_calibration"] is None

required_columns = [
    "unit_uid", "_irwin_norm", "IncidentName", "Year",
    "hold_label", "outer_fold", SCORE_COLUMN,
]
predictions = pd.read_parquet(FORWARD_PREDICTIONS_PATH, columns=required_columns)
assert predictions["unit_uid"].is_unique
assert set(predictions["Year"].unique()) == set(FORWARD_YEARS)
assert predictions[SCORE_COLUMN].between(0.0, 1.0, inclusive="neither").all()
assert np.isfinite(predictions[SCORE_COLUMN].to_numpy(float)).all()

year_audit = (
    predictions.groupby("Year", as_index=False)
    .agg(units=("unit_uid", "size"), incidents=("_irwin_norm", "nunique"), held_rate=("hold_label", "mean"), mean_score=(SCORE_COLUMN, "mean"))
)
for row in year_audit.itertuples(index=False):
    assert row.units == EXPECTED_YEAR_UNITS[int(row.Year)]
    assert row.incidents == EXPECTED_YEAR_INCIDENTS[int(row.Year)]

display(year_audit)
print(f"Loaded {len(predictions):,} forward predictions from {predictions['_irwin_norm'].nunique():,} incidents.")

## Threshold and utility helpers

Positive means predicting **Held** (`hold_label = 1`). A false positive therefore means predicting Held when the observed label is not Held; a false negative means failing to identify an observed Held unit. Cost ratios below are deliberately treated as hypothetical sensitivity scenarios.

In [ ]:
def safe_divide(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan

def classification_metrics(frame, threshold, score_column=SCORE_COLUMN):
    y = frame["hold_label"].to_numpy(np.int8)
    score = frame[score_column].to_numpy(float)
    pred = score >= float(threshold)
    tp = int(np.sum(pred & (y == 1)))
    fp = int(np.sum(pred & (y == 0)))
    tn = int(np.sum(~pred & (y == 0)))
    fn = int(np.sum(~pred & (y == 1)))
    sensitivity = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    precision = safe_divide(tp, tp + fp)
    f1 = safe_divide(2 * tp, 2 * tp + fp + fn)
    pooled_ba = float(np.nanmean([sensitivity, specificity]))

    incident_scores = []
    for _, group in frame.assign(_pred=pred).groupby("_irwin_norm", sort=False):
        gy = group["hold_label"].to_numpy(np.int8)
        if np.unique(gy).size != 2:
            continue
        gp = group["_pred"].to_numpy(bool)
        gtp = np.sum(gp & (gy == 1))
        gfn = np.sum(~gp & (gy == 1))
        gtn = np.sum(~gp & (gy == 0))
        gfp = np.sum(gp & (gy == 0))
        incident_scores.append(0.5 * (gtp / (gtp + gfn) + gtn / (gtn + gfp)))

    metrics = {
        "threshold": float(threshold),
        "units": len(frame),
        "incidents": frame["_irwin_norm"].nunique(),
        "mixed_incidents": len(incident_scores),
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision_ppv": precision,
        "f1": f1,
        "pooled_balanced_accuracy": pooled_ba,
        "mixed_incident_macro_balanced_accuracy": float(np.mean(incident_scores)) if incident_scores else np.nan,
        "predicted_held_rate": float(np.mean(pred)),
    }
    prevalence = float(np.mean(y))
    for scenario, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        average_cost = (cost_fp * fp + cost_fn * fn) / len(frame)
        cost_all_held = cost_fp * (1.0 - prevalence)
        cost_none_held = cost_fn * prevalence
        best_constant_cost = min(cost_all_held, cost_none_held)
        metrics[f"average_cost__{scenario}"] = float(average_cost)
        metrics[f"best_constant_cost__{scenario}"] = float(best_constant_cost)
        metrics[f"relative_cost_reduction__{scenario}"] = float(1.0 - average_cost / best_constant_cost) if best_constant_cost > 0 else np.nan
    return metrics

def threshold_sweep(frame, thresholds=THRESHOLDS):
    return pd.DataFrame([classification_metrics(frame, threshold) for threshold in thresholds])

def select_threshold(sweep, policy):
    if policy == "FIXED_0_5":
        return 0.5, "fixed; no data selection", True
    if policy == "MAX_PAST_MIXED_INCIDENT_BA":
        ranked = sweep.sort_values(
            ["mixed_incident_macro_balanced_accuracy", "pooled_balanced_accuracy", "threshold"],
            ascending=[False, False, True],
        )
        return float(ranked.iloc[0]["threshold"]), "maximize historical mixed-incident macro balanced accuracy", True
    if policy == "PAST_PPV_AT_LEAST_0_85":
        eligible = sweep[(sweep["precision_ppv"] >= PPV_TARGET) & (sweep["tp"] > 0)].copy()
        if len(eligible):
            ranked = eligible.sort_values(["sensitivity", "threshold"], ascending=[False, True])
            return float(ranked.iloc[0]["threshold"]), f"maximize sensitivity subject to historical PPV >= {PPV_TARGET:.2f}", True
        ranked = sweep.sort_values(["precision_ppv", "sensitivity", "threshold"], ascending=[False, False, True])
        return float(ranked.iloc[0]["threshold"]), f"PPV target infeasible; maximize historical PPV below {PPV_TARGET:.2f}", False
    if policy in COST_SCENARIOS:
        metric = f"average_cost__{policy}"
        ranked = sweep.sort_values([metric, "mixed_incident_macro_balanced_accuracy", "threshold"], ascending=[True, False, True])
        cost_fp, cost_fn = COST_SCENARIOS[policy]
        return float(ranked.iloc[0]["threshold"]), f"minimize historical empirical cost with FP:FN={cost_fp:g}:{cost_fn:g}", True
    raise KeyError(policy)

def policy_objective(metrics, policy):
    if policy == "MAX_PAST_MIXED_INCIDENT_BA":
        return "mixed_incident_macro_balanced_accuracy", metrics["mixed_incident_macro_balanced_accuracy"], "higher"
    if policy == "PAST_PPV_AT_LEAST_0_85":
        return "sensitivity_subject_to_ppv_target", metrics["sensitivity"] if metrics["precision_ppv"] >= PPV_TARGET else np.nan, "higher"
    if policy in COST_SCENARIOS:
        name = f"average_cost__{policy}"
        return name, metrics[name], "lower"
    return "fixed_threshold", np.nan, "not_optimized"

print("Threshold grid:", THRESHOLDS[0], "to", THRESHOLDS[-1], "| policies:", len(POLICIES))

## Past-to-future threshold transfer

Each selected threshold is frozen before the evaluation year is inspected. The saved evaluation table therefore represents the deployable analysis. Oracle comparisons in the following section are deliberately separate.

In [ ]:
selection_rows = []
transfer_metric_rows = []
sweep_parts = []
transfer_predictions = predictions[predictions["Year"].isin(EVALUATION_YEARS)][required_columns].copy()

for test_year in EVALUATION_YEARS:
    historical = predictions[predictions["Year"] < test_year].copy()
    test = predictions[predictions["Year"] == test_year].copy()
    assert historical["Year"].max() < test_year
    assert set(historical["_irwin_norm"]).isdisjoint(set(test["_irwin_norm"]))

    historical_sweep = threshold_sweep(historical)
    historical_sweep["scope"] = "historical_selection"
    historical_sweep["test_year"] = test_year
    historical_sweep["selection_start_year"] = int(historical["Year"].min())
    historical_sweep["selection_end_year"] = int(historical["Year"].max())
    sweep_parts.append(historical_sweep)

    for policy in POLICIES:
        threshold, rule, target_feasible = select_threshold(historical_sweep, policy)
        selection_rows.append({
            "test_year": test_year,
            "policy": policy,
            "policy_label": POLICY_LABELS[policy],
            "threshold": threshold,
            "selection_rule": rule,
            "historical_target_feasible": target_feasible,
            "selection_start_year": int(historical["Year"].min()),
            "selection_end_year": int(historical["Year"].max()),
            "selection_units": len(historical),
            "selection_incidents": historical["_irwin_norm"].nunique(),
        })
        metrics = classification_metrics(test, threshold)
        transfer_metric_rows.append({
            "test_year": test_year,
            "policy": policy,
            "policy_label": POLICY_LABELS[policy],
            "selection_end_year": int(historical["Year"].max()),
            **metrics,
        })
        row_mask = transfer_predictions["Year"].eq(test_year)
        transfer_predictions.loc[row_mask, f"predicted_held__{policy}"] = (
            transfer_predictions.loc[row_mask, SCORE_COLUMN].to_numpy(float) >= threshold
        ).astype(np.int8)

selections = pd.DataFrame(selection_rows)
transfer_metrics = pd.DataFrame(transfer_metric_rows)
all_sweeps = pd.concat(sweep_parts, ignore_index=True)
for policy in POLICIES:
    transfer_predictions[f"predicted_held__{policy}"] = transfer_predictions[f"predicted_held__{policy}"].astype(np.int8)

selections.to_csv(OUTPUT_ROOT / "LEAKAGE_SAFE_SELECTED_THRESHOLDS_V1.csv", index=False)
transfer_metrics.to_csv(OUTPUT_ROOT / "LEAKAGE_SAFE_TRANSFER_METRICS_V1.csv", index=False)
all_sweeps.to_parquet(PROCESSED_ROOT / "HISTORICAL_THRESHOLD_SWEEPS_V1.parquet", index=False)
transfer_predictions.to_parquet(PROCESSED_ROOT / "LIGHTGBM_TEMPORAL_DECISION_PREDICTIONS_V1.parquet", index=False)

display(selections[["test_year", "policy_label", "threshold", "selection_start_year", "selection_end_year", "historical_target_feasible"]])
display(transfer_metrics[["test_year", "policy_label", "threshold", "sensitivity", "specificity", "precision_ppv", "mixed_incident_macro_balanced_accuracy", "predicted_held_rate"]])

In [ ]:
oracle_rows = []
for test_year in EVALUATION_YEARS:
    test = predictions[predictions["Year"] == test_year].copy()
    test_sweep = threshold_sweep(test)
    for policy in [p for p in POLICIES if p != "FIXED_0_5"]:
        oracle_threshold, oracle_rule, oracle_feasible = select_threshold(test_sweep, policy)
        oracle_metrics = classification_metrics(test, oracle_threshold)
        transfer_row = transfer_metrics[(transfer_metrics["test_year"] == test_year) & (transfer_metrics["policy"] == policy)].iloc[0].to_dict()
        objective_name, oracle_value, direction = policy_objective(oracle_metrics, policy)
        _, transfer_value, _ = policy_objective(transfer_row, policy)
        if np.isfinite(oracle_value) and np.isfinite(transfer_value):
            regret = oracle_value - transfer_value if direction == "higher" else transfer_value - oracle_value
        else:
            regret = np.nan
        oracle_rows.append({
            "test_year": test_year,
            "policy": policy,
            "policy_label": POLICY_LABELS[policy],
            "transferred_threshold": float(transfer_row["threshold"]),
            "oracle_threshold": oracle_threshold,
            "threshold_delta_oracle_minus_transfer": oracle_threshold - float(transfer_row["threshold"]),
            "objective_name": objective_name,
            "objective_direction": direction,
            "transferred_objective": transfer_value,
            "oracle_objective": oracle_value,
            "transfer_regret": regret,
            "oracle_target_feasible": oracle_feasible,
            "oracle_rule_diagnostic_only": oracle_rule,
        })

oracle_diagnostic = pd.DataFrame(oracle_rows)
oracle_diagnostic.to_csv(OUTPUT_ROOT / "WITHIN_YEAR_ORACLE_DIAGNOSTIC_ONLY_V1.csv", index=False)

stability = (
    selections.groupby(["policy", "policy_label"], as_index=False)["threshold"]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)
stability["range"] = stability["max"] - stability["min"]
stability.to_csv(OUTPUT_ROOT / "THRESHOLD_STABILITY_SUMMARY_V1.csv", index=False)

cost_rows = []
for policy in COST_SCENARIOS:
    metric = f"relative_cost_reduction__{policy}"
    for row in transfer_metrics[transfer_metrics["policy"] == policy].itertuples(index=False):
        cost_rows.append({
            "test_year": int(row.test_year),
            "policy": policy,
            "policy_label": POLICY_LABELS[policy],
            "threshold": float(row.threshold),
            "relative_cost_reduction_vs_best_constant": float(getattr(row, metric)),
            "average_cost": float(getattr(row, f"average_cost__{policy}")),
            "best_constant_cost": float(getattr(row, f"best_constant_cost__{policy}")),
        })
cost_summary = pd.DataFrame(cost_rows)
cost_summary.to_csv(OUTPUT_ROOT / "COST_POLICY_TRANSFER_SUMMARY_V1.csv", index=False)

display(stability)
display(oracle_diagnostic)
display(cost_summary)

In [ ]:
decision_rows = []
for test_year in FORWARD_YEARS:
    year_frame = predictions[predictions["Year"] == test_year]
    y = year_frame["hold_label"].to_numpy(np.int8)
    score = year_frame[SCORE_COLUMN].to_numpy(float)
    prevalence = float(np.mean(y))
    n = len(y)
    for threshold in DECISION_THRESHOLDS:
        pred = score >= threshold
        tp = int(np.sum(pred & (y == 1)))
        fp = int(np.sum(pred & (y == 0)))
        odds = threshold / (1.0 - threshold)
        decision_rows.append({
            "test_year": test_year,
            "threshold_probability": threshold,
            "model_net_benefit": tp / n - fp / n * odds,
            "all_held_net_benefit": prevalence - (1.0 - prevalence) * odds,
            "none_held_net_benefit": 0.0,
        })
decision_curve = pd.DataFrame(decision_rows)
decision_curve.to_csv(OUTPUT_ROOT / "TEMPORAL_DECISION_CURVES_V1.csv", index=False)
print("Decision-curve rows:", len(decision_curve))
print("Caution: net benefit uses raw frozen scores; interpret absolute threshold probabilities in light of 05H temporal calibration drift.")

## Publication figures and final audit

The figures emphasize transfer behavior. Any apparent oracle advantage is kept in tabular diagnostics rather than visually promoted as an achievable deployment result.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5), constrained_layout=True)
for policy in POLICIES:
    group = selections[selections["policy"] == policy].sort_values("test_year")
    ax.plot(group["test_year"], group["threshold"], marker="o", linewidth=2, color=COLORS[policy], label=POLICY_LABELS[policy])
ax.set_xticks(EVALUATION_YEARS)
ax.set_ylim(0, 1)
ax.set_xlabel("Forward evaluation year")
ax.set_ylabel("Threshold selected using prior forward predictions")
ax.set_title("Frozen LightGBM threshold stability under temporal transfer")
ax.grid(alpha=0.25)
ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.savefig(FIGURE_ROOT / "FIG_06C_01_TEMPORAL_THRESHOLD_STABILITY_V1.png", dpi=240, bbox_inches="tight")
fig.savefig(FIGURE_ROOT / "FIG_06C_01_TEMPORAL_THRESHOLD_STABILITY_V1.pdf", bbox_inches="tight")
plt.show()

metric_specs = [
    ("sensitivity", "Sensitivity"),
    ("specificity", "Specificity"),
    ("precision_ppv", "Precision (PPV)"),
    ("mixed_incident_macro_balanced_accuracy", "Mixed-incident macro balanced accuracy"),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, constrained_layout=True)
for ax, (metric, label) in zip(axes.ravel(), metric_specs):
    for policy in POLICIES:
        group = transfer_metrics[transfer_metrics["policy"] == policy].sort_values("test_year")
        ax.plot(group["test_year"], group[metric], marker="o", linewidth=1.8, color=COLORS[policy], label=POLICY_LABELS[policy])
    ax.set_title(label)
    ax.set_xticks(EVALUATION_YEARS)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
fig.supxlabel("Forward evaluation year")
fig.suptitle("Leakage-safe temporal transfer of LightGBM decision policies", fontsize=15)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, -0.04), ncol=4)
fig.savefig(FIGURE_ROOT / "FIG_06C_02_TEMPORAL_TRANSFER_METRICS_V1.png", dpi=240, bbox_inches="tight")
fig.savefig(FIGURE_ROOT / "FIG_06C_02_TEMPORAL_TRANSFER_METRICS_V1.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for policy in COST_SCENARIOS:
    group = cost_summary[cost_summary["policy"] == policy].sort_values("test_year")
    ax.plot(group["test_year"], group["relative_cost_reduction_vs_best_constant"], marker="o", linewidth=2, color=COLORS[policy], label=POLICY_LABELS[policy])
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_xticks(EVALUATION_YEARS)
ax.set_xlabel("Forward evaluation year")
ax.set_ylabel("Relative cost reduction vs best constant action")
ax.set_title("Transferred cost policies under explicit error-cost scenarios")
ax.grid(alpha=0.25)
ax.legend(frameon=False, fontsize=9)
fig.savefig(FIGURE_ROOT / "FIG_06C_03_COST_POLICY_VALUE_V1.png", dpi=240, bbox_inches="tight")
fig.savefig(FIGURE_ROOT / "FIG_06C_03_COST_POLICY_VALUE_V1.pdf", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, constrained_layout=True)
for ax, test_year in zip(axes.ravel(), FORWARD_YEARS):
    group = decision_curve[decision_curve["test_year"] == test_year]
    ax.plot(group["threshold_probability"], group["model_net_benefit"], color="#4c72b0", linewidth=2, label="Frozen LightGBM")
    ax.plot(group["threshold_probability"], group["all_held_net_benefit"], color="#8c8c8c", linestyle="--", label="Classify all Held")
    ax.axhline(0, color="black", linewidth=1, label="Classify none Held")
    ax.set_title(f"Forward test year {test_year}")
    ax.set_xlabel("Threshold probability")
    ax.set_ylabel("Net benefit")
    ax.grid(alpha=0.25)
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Temporal decision curves for the frozen LightGBM score", fontsize=15)
fig.savefig(FIGURE_ROOT / "FIG_06C_04_TEMPORAL_DECISION_CURVES_V1.png", dpi=240, bbox_inches="tight")
fig.savefig(FIGURE_ROOT / "FIG_06C_04_TEMPORAL_DECISION_CURVES_V1.pdf", bbox_inches="tight")
plt.show()

In [ ]:
label_columns = [f"predicted_held__{policy}" for policy in POLICIES]
checks = pd.DataFrame([
    {"check": "upstream_05h_checks_pass", "pass": bool(upstream_pass)},
    {"check": "frozen_05c_feature_contract_preserved", "pass": upstream_contract["feature_set"] == "E_FULL_EXTENDED" and upstream_contract["feature_count"] == 803},
    {"check": "no_model_training_in_06c", "pass": True},
    {"check": "exact_forward_years_loaded", "pass": set(predictions["Year"].unique()) == set(FORWARD_YEARS)},
    {"check": "exact_forward_year_counts", "pass": all(len(predictions[predictions["Year"] == year]) == EXPECTED_YEAR_UNITS[year] for year in FORWARD_YEARS)},
    {"check": "all_forward_scores_finite_and_bounded", "pass": bool(np.isfinite(predictions[SCORE_COLUMN]).all() and predictions[SCORE_COLUMN].between(0, 1, inclusive="neither").all())},
    {"check": "2021_reserved_as_warmup_only", "pass": 2021 not in set(selections["test_year"])},
    {"check": "all_threshold_selection_strictly_precedes_test_year", "pass": bool((selections["selection_end_year"] < selections["test_year"]).all())},
    {"check": "exact_policy_year_selections", "pass": len(selections) == len(POLICIES) * len(EVALUATION_YEARS)},
    {"check": "exact_policy_year_transfer_metrics", "pass": len(transfer_metrics) == len(POLICIES) * len(EVALUATION_YEARS)},
    {"check": "all_transfer_labels_complete", "pass": bool(transfer_predictions[label_columns].notna().all().all())},
    {"check": "all_selected_thresholds_on_declared_grid", "pass": bool(selections["threshold"].isin(THRESHOLDS).all())},
    {"check": "oracle_results_explicitly_diagnostic_only", "pass": bool(oracle_diagnostic["oracle_rule_diagnostic_only"].notna().all())},
    {"check": "decision_curve_complete", "pass": len(decision_curve) == len(FORWARD_YEARS) * len(DECISION_THRESHOLDS)},
    {"check": "cost_scenarios_explicit_and_complete", "pass": len(cost_summary) == len(COST_SCENARIOS) * len(EVALUATION_YEARS)},
    {"check": "thresholds_reported_as_policies_not_universal_optimum", "pass": True},
    {"check": "paper_primary_remains_frozen_lightgbm", "pass": True},
])
PASS = bool(checks["pass"].all())
checks.to_csv(OUTPUT_ROOT / "FINAL_LIGHTGBM_TEMPORAL_DECISION_POLICY_CHECKS_V1.csv", index=False)

def md_table(frame):
    try:
        return frame.to_markdown(index=False)
    except Exception:
        return frame.to_string(index=False)

report_lines = [
    "# 06C Frozen LightGBM Temporal Decision-Policy Audit",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    f"Generated UTC: {datetime.now(timezone.utc).isoformat()}",
    "",
    "## Scientific status",
    "",
    "Post-freeze decision-policy analysis using saved 05H expanding-window predictions. No model was trained, tuned, calibrated, or replaced in 06C.",
    "",
    "Thresholds for each evaluation year were selected exclusively from earlier forward-prediction years. The 2021 predictions form the warm-up history. Within-year oracle thresholds are diagnostic upper bounds only.",
    "",
    "Cost ratios are hypothetical sensitivity scenarios and require stakeholder elicitation before operational use. Decision-curve threshold probabilities should be interpreted cautiously because 05H documented temporal calibration drift.",
    "",
    "## Input year audit",
    "",
    md_table(year_audit),
    "",
    "## Leakage-safe selected thresholds",
    "",
    md_table(selections),
    "",
    "## Past-to-future transfer metrics",
    "",
    md_table(transfer_metrics),
    "",
    "## Threshold stability",
    "",
    md_table(stability),
    "",
    "## Explicit cost-scenario transfer",
    "",
    md_table(cost_summary),
    "",
    "## Within-year oracle diagnostic only",
    "",
    md_table(oracle_diagnostic),
    "",
    "## Final checks",
    "",
    md_table(checks),
    "",
    "## Decision",
    "",
    "`PASS / TEMPORAL DECISION-POLICY AUDIT COMPLETE; NO UNIVERSAL THRESHOLD CLAIMED`" if PASS else "`REVIEW REQUIRED`",
]
report_path = OUTPUT_ROOT / "LIGHTGBM_TEMPORAL_DECISION_POLICY_REPORT_V1.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

display(checks)
print("\n06C:", "PASS" if PASS else "REVIEW REQUIRED")
print("Saved report:", report_path)
print("DONE:", NOTEBOOK_BUILD_ID)